# Running the study on Colab

Everything below assumes an **A100** runtime and a Drive folder holding the
preprocessed data. Run the cells in order; each one prints enough to tell you
whether to continue.

## Before you start

**1. Set the runtime.** Runtime -> Change runtime type -> **A100 GPU**.
Cell 1 refuses to continue on a T4, because the timings below assume A100 and a
T4 would turn a 3-hour study into an overnight one.

**2. Put two things on Drive**, at `MyDrive/ecg/`:

| what | from | size |
|---|---|---|
| `store_100hz/` | your local `data/store_100hz/` | 500 MB |
| `ptbxl/ptbxl_database.csv` | your local `data/ptbxl/` | 6.6 MB |
| `ptbxl/scp_statements.csv` | your local `data/ptbxl/` | 10 KB |

**Do not upload the 3 GB `records100/` and `records500/` trees.** The waveforms
are already inside the store; only the two CSVs are read at training time.

Upload the store as a **folder**, not as a zip you unpack on Drive — unzipping
onto a FUSE mount is far slower than uploading the two `.npy` files directly.

**3. Cell 4 copies the data to `/content`.** Drive holds the master copy;
training reads the local one. This is not premature optimisation: the store is
*memory-mapped*, so reading it from Drive turns every page fault into a network
round-trip, and a mount that goes stale mid-run surfaces as a SIGBUS inside
numpy rather than a catchable error. You pay the same 500 MB read either way —
copying makes it one bulk sequential read, which is what Drive is fastest at,
and every later `ecg-run` in the session reads from local disk instead.

## The order to run in

1. **Cells 1-4** — setup and checks. Fast.
2. **Cell 5, the dry run** — prints the 14-run grid without booking anything.
   Read it. This is the cheapest place to catch a wrong fraction.
3. **Cell 6, the timing probe** — one epoch, to turn "about 3 hours" into a
   real number before you commit to the full study.
4. **Cell 7, the mask-ratio ablation** — 3 pretrains + 3 short fine-tunes,
   selected on validation. Pick the winner and pass it to cell 8.
5. **Cell 8, the study** — the 14 runs.
6. **Cells 9-10** — results.

## If Colab disconnects

Re-run cells 1-4, then re-run the same training cell. **Finished runs are
skipped**: each writes `result.json` when it completes, and the runner resumes
from the one that was interrupted. You lose at most the run that was in flight.

## Where things live

- **Checkpoints and results** go to Drive, under `MyDrive/ecg/runs/<run name>/`.
- **MLflow's database stays on local disk** at `/content/mlruns/mlflow.db`, and
  a consistent copy is written to Drive every epoch. This is deliberate:
  SQLite's locking assumes POSIX semantics a Drive mount does not honour, so a
  database living on Drive can corrupt silently. Cell 8 warns if you override
  `--tracking` with a Drive path.

## 1. Check the GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> A100."
name = torch.cuda.get_device_name(0)
memory = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}, {memory:.0f} GB, torch {torch.__version__}")

if "A100" not in name:
    print(
        f"\nWARNING: this is a {name}, not an A100. The timings in this "
        "notebook assume an A100; expect roughly 3-4x longer on a T4, and "
        "bf16 autocast is only useful on Ampere or newer."
    )

## 2. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

# Drive holds the master data and receives every checkpoint.
ECG = Path("/content/drive/MyDrive/ecg")
DRIVE_STORE = ECG / "store_100hz"
DRIVE_METADATA = ECG / "ptbxl"
RUNS = ECG / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

# Training reads from local disk. See cell 4.
LOCAL = Path("/content/data")
STORE = LOCAL / "store_100hz"
METADATA = LOCAL / "ptbxl"

# Which version of the architecture this session is running. Change it whenever
# you change the model -- see section 8. A slug, no spaces: it becomes part of
# every run name and every output directory.
VARIANT = "base"

print(f"drive:  {ECG}")
print(f"local:  {LOCAL}")
print(f"variant: {VARIANT}")

## 3. Install the package

In [ ]:
!git clone --depth 1 https://github.com/GilAmihai77/ECG-ML.git /content/ECG-ML 2>/dev/null || (cd /content/ECG-ML && git pull -q)
%pip install -q -e /content/ECG-ML

# The editable install writes a .pth file into site-packages, but `site` only
# reads .pth files when the interpreter starts, so it does not affect the kernel
# that just ran the install. Point at src/ by hand instead of restarting.
# Subprocesses (the ecg-run console script) start fresh and need none of this.
import importlib
import sys

if "/content/ECG-ML/src" not in sys.path:
    sys.path.insert(0, "/content/ECG-ML/src")
importlib.invalidate_caches()

import ecg

print("ecg", ecg.__version__, "from", ecg.__file__)

## 4. Copy the data to local disk, then check it

The store is memory-mapped, so leaving it on Drive would turn page faults into
network round-trips during training and make a stale mount a SIGBUS inside
numpy. Copy once, ~30-60 s, then everything reads locally at SSD speed.

The check afterwards prints the cohort sizes. If they do not match what you saw
locally, stop -- a truncated upload is much cheaper to find now than an hour
into pretraining.

In [ ]:
import shutil
import time

for source, destination in ((DRIVE_STORE, STORE), (DRIVE_METADATA, METADATA)):
    assert source.exists(), f"missing on Drive: {source}"
    if destination.exists():
        print(f"{destination} already present, skipping copy")
        continue
    start = time.perf_counter()
    shutil.copytree(source, destination)
    size = sum(f.stat().st_size for f in destination.rglob("*") if f.is_file())
    print(
        f"copied {source.name}: {size / 1e6:.0f} MB in "
        f"{time.perf_counter() - start:.0f} s"
    )

!df -h /content | tail -1

In [ ]:
from ecg.data.datasets import assert_patient_disjoint, build_cohorts, describe_cohorts
from ecg.data.preprocess import WaveformStore
from ecg.data.ptbxl import load_metadata

for required in (STORE / "waveforms.npy", METADATA / "ptbxl_database.csv"):
    assert required.exists(), f"missing locally: {required}"

store = WaveformStore.load(STORE)
metadata = load_metadata(METADATA)
cohorts = build_cohorts(metadata)

print(f"store: {len(store)} records, {store.n_samples} samples, {store.sampling_rate} Hz")
assert len(store) == 21799, f"expected 21799 records, got {len(store)}"

assert_patient_disjoint(cohorts, metadata)
print("patient-disjoint: ok\n")
print(describe_cohorts(cohorts).to_string())

### Label distribution across the splits

Folds 1-8 as `train`, fold 9 as `val`, fold 10 as `test`. Bars are the
percentage of each split carrying a label, so the splits stay comparable
despite differing in size by roughly five to one; the absolute count sits at the
end of every bar.

The labels are multi-label, so the bars within a split sum past 100% and a
record carrying no scorable superclass contributes to none of them. `unlabelled`
in the table below is that count -- those records are retained and reported, not
dropped (integrity rule 8).

In [ ]:
from ecg.data.ptbxl import SUPERCLASSES
from ecg.experiments.report import label_distribution, plot_label_distribution

figure = plot_label_distribution(cohorts)

distribution = label_distribution(cohorts)
counts = distribution[[f"{c}_n" for c in SUPERCLASSES]].astype(int)
counts.columns = SUPERCLASSES
percents = distribution[[f"{c}_pct" for c in SUPERCLASSES]].round(1)
percents.columns = SUPERCLASSES

print("positives")
print(counts.to_string())
print("\nprevalence (% of split)")
print(percents.to_string())
print("\nrecords", distribution["n_records"].astype(int).to_dict())
print("labels per record", distribution["labels_per_record"].round(2).to_dict())
print("unlabelled", distribution["unlabelled"].astype(int).to_dict())

#### What to notice

**Two kinds of imbalance, and they are different problems.**

1. **Between classes.** NORM is roughly four to five times HYP in every split.
   This is why macro averaging is used throughout and why F1 needs a per-class
   threshold -- a single 0.5 cutoff optimises for NORM and quietly gives up on
   HYP. Macro is *unweighted* (`metrics._macro`), so HYP counts as much as NORM.

2. **Between train and the held-out folds.** Train is about 51% NORM where val
   and test are about 43%. That gap is **not** in the published folds -- raw,
   all three sit near 43.7% -- it is created by the exclusion policy, which
   drops far more from train than from the held-out splits:

   | dropped from | train | val | test |
   |---|---|---|---|
   | not validated by human | 5,743 | 0 | 0 |
   | duplicate patient record | 1,006 | 0 | 0 |
   | **retained** | **58.9%** | **96.3%** | **95.8%** |

   A third of the training folds are machine-labelled and none of folds 9-10
   are, so "drop machine labels" only ever bites train. Those records skew
   pathological, and removing them leaves train more NORM-heavy than test.

This asymmetry is deliberate, not a leak -- see `apply_supervised_filter`.
Cohort-shaping rules applied to the held-out folds would make the headline
metric describe a cleaner population than the real one. The cost is a genuine
train/test shift, and it is worth stating when reporting results rather than
discovering it in a question.

## 5. Dry run: print the plan

Nothing is trained here. Read the grid before committing GPU time.

In [ ]:
!ecg-run --dry-run \
    --store "$STORE" --metadata "$METADATA" --output "$RUNS/study" \
    --variant $VARIANT \
    --fractions 0.2 0.5 1.0 --epochs 50 --d-model 256 --n-heads 8

## 6. Timing probe

One epoch of each kind, so the full study's cost is a measurement rather than a
guess. It writes to a throwaway directory, so it does not pollute the real
results or the resume state.

Multiply what you see by `--epochs` to size the study.

In [ ]:
import time

start = time.perf_counter()
!ecg-run \
    --store "$STORE" --metadata "$METADATA" \
    --output /content/timing --tracking /content/timing/mlflow.db \
    --epochs 1 --fractions 1.0 --no-track
print(f"\nprobe wall clock: {(time.perf_counter() - start) / 60:.1f} min")

### Reading the probe

The probe ran 2 pretraining epochs and 4 supervised epochs at 100% labels.
For a study with `E` epochs:

- pretraining is `2 x E` epochs
- the supervised grid is `E x (0.2 + 0.2 + 0.5 + 0.5 + 1.0 + 1.0) x 2 embedders`
  = `6.8 x E` supervised-epoch-equivalents at full size

so the study costs roughly `2 x E` pretraining epochs plus `6.8 x E` fine-tuning
epochs. At 50 epochs that is 100 pretraining epochs and ~340 fine-tuning
epoch-equivalents. If the probe says a pretraining epoch takes 40 s, budget
about 70 minutes for pretraining alone.

## 7. Mask-ratio ablation

Three pretraining runs at ratios 0.3 / 0.5 / 0.7, each fine-tuned at the
smallest label fraction -- where SSL's effect is largest and the runs are
cheapest. **Selection is on validation.**

Run this before the study, and pass the winning ratio to cell 8. Use the same
ratio for every arm: tuning the mask per arm would fold the SSL setup into what
is being compared.

In [ ]:
!ecg-run --plan ablation \
    --store "$STORE" --metadata "$METADATA" \
    --output "$RUNS/ablation" --tracking /content/mlruns/mlflow.db \
    --experiment ecg-ablation --variant $VARIANT \
    --mask-ratios 0.3 0.5 0.7 --fractions 0.2 --epochs 30

In [ ]:
from ecg.experiments.runner import load_results

# load_results, not results.csv: the CSV holds only the plan that last ran, so
# a second variant's ablation would overwrite the first's summary.
ablation = load_results(RUNS / "ablation")
table = (
    ablation[
        (ablation["kind"] == "supervised") & (ablation["variant"] == VARIANT)
    ]
    .loc[:, ["mask_ratio", "val_macro_auroc", "test_macro_auroc"]]
    .sort_values("mask_ratio")
)
print(table.to_string(index=False))
print(
    "\nselect on val_macro_auroc:",
    table.loc[table["val_macro_auroc"].idxmax(), "mask_ratio"],
)

## 8. The study

Fourteen runs: two pretraining runs, then four arms at each of three label
fractions. **Set `MASK_RATIO` to whatever cell 7 selected.**

Safe to re-run after a disconnect -- finished runs are skipped.

### `VARIANT` -- set in cell 2, change it whenever you change the model

A run name says which embedder and which label fraction -- nothing about the
architecture. A run name is also its output directory, and finished directories
are skipped. So if you change the transformer and re-run with the same
`VARIANT`, **all fourteen runs are skipped and you get the old architecture's
numbers back**, labelled as the new one's.

`--variant` prefixes every run name, which gives the new architecture its own
directories and its own rows in MLflow. Name what changed, as a slug with no
spaces:

| you changed | `VARIANT` |
|---|---|
| nothing yet, the first study | `base` |
| 4 layers -> 6 | `deep6` |
| one more conv layer in the stem | `extra-conv-layer` |
| d_model 256 -> 384 | `d384` |

Keep `--experiment ecg-ssl` the same across all of them. Old and new belong in
one table -- that comparison is the whole point, and separate experiments would
turn it into a manual join.

The run also logs the git commit and whether the tree was dirty, because the
variant is a label you type and the commit is not. **Commit before you run**: a
dirty tree means the SHA names something that is not what ran, and the run will
say so in its `git_dirty` tag.

If you re-run across a code change without moving the variant, the runner
prints the commit each skipped run came from and warns at the end. That warning
means the results on screen are the old model's.

In [ ]:
MASK_RATIO = 0.5  # <- from cell 7;  VARIANT is set in cell 2

!ecg-run \
    --store "$STORE" --metadata "$METADATA" \
    --output "$RUNS/study" --tracking /content/mlruns/mlflow.db \
    --experiment ecg-ssl --variant $VARIANT \
    --mask-ratio $MASK_RATIO --mask-span 2 \
    --fractions 0.2 0.5 1.0 \
    --epochs 50 --batch-size 256 --d-model 256 --n-layers 4 --n-heads 8

## 9. Results

`load_results` rather than `results.csv`: the CSV holds only the plan that last
ran, so a second variant's study overwrites the first's summary. The per-run
`result.json` files are never overwritten -- each variant has its own
directories -- so this rebuilds the full table from them, **every variant
included**.

When more than one variant is present the tables split by it instead of
averaging across it. Two architectures averaged into one row would be a number
that describes neither.

In [ ]:
from ecg.experiments.runner import label_efficiency_table, load_results, ssl_benefit

frame = load_results(RUNS / "study")
print(frame.groupby("variant")["run"].count().to_string(), "\n")

print("label efficiency (test macro AUROC)")
print(label_efficiency_table(frame).to_string())

print("\nwhat SSL bought")
print(ssl_benefit(frame).to_string())

print("\nembedder comparison at each fraction (test macro AUROC)")
print(
    frame[frame["kind"] == "supervised"]
    .pivot_table(
        # variant first, always: A/B and C/D are only valid within one
        # architecture, so the level has to be visible even when there is
        # currently only one.
        index=["variant", "label_fraction", "pretrained"],
        columns="embedder",
        values="test_macro_auroc",
    )
    .to_string()
)

## 10. The label-efficiency curve

The study's headline figure: does the SSL gap widen as labels get scarcer?

**One architecture per figure.** The four arms are only comparable within a
variant, so the cell plots `VARIANT` from cell 8; change it and re-run to see
another. Overlaying two architectures here would put eight lines on two panels
and make the SSL gap -- the thing the figure exists to show -- the hardest thing
on it to see.

In [ ]:
import matplotlib.pyplot as plt

curve = label_efficiency_table(frame)
if "variant" in (curve.index.names or []):
    curve = curve.loc[VARIANT]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for axis, embedder in zip(axes, ("linear", "conv")):
    for arm, style in ((embedder, "o--"), (f"{embedder}+ssl", "o-")):
        if arm in curve:
            axis.plot(curve.index * 100, curve[arm], style, label=arm)
    axis.set_title(f"{embedder} embedder")
    axis.set_xlabel("labelled training data (%)")
    axis.grid(alpha=0.3)
    axis.legend()

axes[0].set_ylabel("test macro AUROC")
fig.suptitle(f"Does SSL help more when labels are scarce?  ({VARIANT})")
fig.tight_layout()
plt.show()

## 11. Browsing the MLflow runs

The database is at `/content/mlruns/mlflow.db` with a per-epoch copy in each
run's output directory on Drive. To browse it locally, download the copy from
the last completed run and point the UI at it:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

To keep the tracking database across sessions, copy it to Drive once at the end
of a session and restore it at the start of the next:

```python
from ecg.training.checkpoints import restore_tracking, sync_tracking

sync_tracking("/content/mlruns/mlflow.db", ECG / "mlflow.db")      # end of session
restore_tracking(ECG / "mlflow.db", "/content/mlruns/mlflow.db")   # start of next
```

## 12. Research summary

Everything below rebuilds the detail `results.csv` does not carry. A run's
`result.json` holds three macro numbers and per-class AUROC; accuracy,
precision, recall, per-class F1 and PR **curves** are not in it, and an average
precision cannot be turned back into a curve. So this section reloads each run's
selected checkpoint and scores validation and test again.

Two properties are preserved on the way:

- **Thresholds are fitted on validation and applied unchanged to test**
  (integrity rule 2), exactly as `_run_supervised` did. They are refitted rather
  than read back, because the checkpoint does not actually store them.
- **The checkpoint reloaded is the one validation selected**, not the last
  epoch's -- the same reload `_run_supervised` performs before touching test.

**Is macro weighted? No.** `macro` is a plain unweighted mean over the classes
with a defined value (`metrics._macro`). That is deliberate: HYP is the rarest
superclass, and support weighting would let NORM hide a failure on it. Every
table below prints a support-`weighted` row next to `macro` so the difference is
visible rather than assumed.

In [ ]:
from collections import Counter

from ecg.experiments import Workspace
from ecg.experiments.report import (
    collect_predictions,
    comparison_table,
    per_class_table,
    plot_pr_curves,
    plot_training_curves,
    style_table,
    training_history,
)

STUDY = RUNS / "study"

# Reuses the store and cohorts already loaded in cell 4 -- the store is 523 MB
# and re-reading it per run would dominate the scoring pass.
workspace = Workspace(store=store, cohorts=cohorts)

# Every finished supervised run in the directory, across all variants. Each one
# means reloading a checkpoint and scoring two cohorts, so this is where a
# second architecture doubles the wait. To look at one only:
#     runs = [r for r in runs if r.variant == VARIANT]
runs = collect_predictions(STUDY, workspace)
print(f"\n{len(runs)} supervised runs scored")
print("by variant:", dict(Counter(r.variant for r in runs)))

### 12a. Metric comparison

One row per run, macro-averaged. `exact_match` is subset accuracy -- the
fraction of records whose five labels are *all* correct.

Read `accuracy` with suspicion. These labels are imbalanced, so per-class
accuracy is high for a model that predicts the majority class and says nothing
the F1 column does not say better. It is here because it was asked for, not
because it should drive a conclusion.

In [ ]:
table = comparison_table(runs, split="test")
style_table(table)

### 12b. Per class

Macro hides which class moved. `f1` below; swap `metric=` for `"precision"`,
`"recall"`, `"accuracy"`, `"auroc"` or `"pr_auc"`.

In [ ]:
for metric in ("f1", "precision", "recall"):
    print(f"\n{metric} by class (test)")
    print(per_class_table(runs, split="test", metric=metric).round(3).to_string())

In [ ]:
# Full detail for one run: every class, every metric, both averagings.
chosen = max(runs, key=lambda r: r.table("test").loc["macro", "auroc"])
print(f"{chosen.name}   (best test macro AUROC)")
style_table(chosen.table("test"))

### 12c. Precision-recall curves

One panel per run, one line per class, average precision in the legend.

The dotted horizontal line in each panel is that class's prevalence -- the
precision a random classifier reaches. PR curves are not comparable across
classes without it: NORM's baseline sits far above HYP's, so the same precision
is a failure for one and a win for the other.

In [ ]:
figure = plot_pr_curves(runs, split="test", ncols=4)
figure.savefig(ECG / "pr_curves.png", dpi=150, bbox_inches="tight")
print("saved", ECG / "pr_curves.png")

### 12d. Training curves

From the MLflow database, which is the only place per-epoch values live --
`result.json` keeps the selected epoch, not the history.

The database is copied off Drive first and opened read-only. SQLite's locking
assumes POSIX semantics a FUSE mount does not honour, and writing to a database
on Drive is how it corrupts silently.

**On `val_loss`:** runs completed before this section was added logged
`train_loss` and the validation *metrics*, but no validation loss -- so that
panel will be empty for them and the cell after it falls back to macro AUROC.
Validation loss is logged from now on (`loops.train_supervised`, computed from
scores that were already in hand; nothing selects on it). Re-run a study to
populate it.

In [ ]:
!cp -f /content/mlruns/mlflow.db /content/mlflow_read.db

history = training_history("/content/mlflow_read.db", experiment="ecg-ssl")
print(sorted(history["key"].unique()))

# Compare the four arms at the scarcest label fraction, where SSL should matter most.
scarcest = min(r.label_fraction for r in runs)
arms = [r.name for r in runs if r.label_fraction == scarcest]
figure = plot_training_curves(
    history, arms, keys=("train_loss", "val_loss"),
    ylabels=("BCE loss (train)", "BCE loss (val)"),
)
figure.savefig(ECG / "training_curves.png", dpi=150, bbox_inches="tight")

In [ ]:
# Overfitting check that works on the runs you already have: training loss
# against the metric selection actually used.
figure = plot_training_curves(
    history, arms, keys=("train_loss", "val_macro_auroc"),
    ylabels=("BCE loss (train)", "macro AUROC (val)"),
)

### 12e. Pretraining curves

The one place a genuine train-vs-validation loss pair already exists: SSL logs
`ssl_loss` on the pool and `holdout_loss` on a slice held out of the pool. A gap
opening between them is the reconstruction task being memorised.

In [ ]:
pretrains = sorted(history.loc[history["key"] == "holdout_loss", "run"].unique())
if pretrains:
    plot_training_curves(
        history, pretrains, keys=("ssl_loss", "holdout_loss"),
        ylabels=("reconstruction MSE (pool)", "reconstruction MSE (holdout)"),
    )
else:
    print("no pretraining runs in this database")